# Appendix — Data Verification Figures

Supplementary figures for the data characterisation section (sec1).
Saved to `thesis_figures/sec7/`.

In [ ]:
import sys, os

os.chdir("/home/bobby/repos/latent-neural-dynamics-modeling")
sys.path.insert(0, ".")
sys.path.insert(0, "notebooks")

In [ ]:
from collections import namedtuple, defaultdict
from pathlib import Path
import numpy as np
import polars as pl
import yaml
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.lines import Line2D
from scipy.signal import welch
from scipy.stats import mannwhitneyu, gaussian_kde

from modules.style import (
    COLOR_DBS_OFF,
    COLOR_DBS_ON,
    COLOR_DPAD,
    COLOR_PSID,
    apply_modules.style,
    hex_to_rgba,
    panel_label,
    stack_bar_label,
)

apply_modules.style()

In [ ]:
OUT = Path("thesis_figures/sec7")
OUT.mkdir(parents=True, exist_ok=True)
results_root = Path("results").resolve()
raw_data_root = Path(
    "resampled_recordings/participants_at_200Hz_scaled_1e6_narrow_band"
)
ECOG_CHANNELS_LABELS = ["ECOG_1", "ECOG_2", "ECOG_3", "ECOG_4"]
p2_root = Path("data/participants_2")

In [ ]:
# Session registry — auto-discovered via discover_session_run.
from modules.loaders import (
    discover_session_run,
    EXP_BEHAVIORAL,
    SESSIONS as _SESSION_NAMES,
)

SessionSpec = namedtuple(
    "SessionSpec", ["label", "participant", "session", "psid_variant", "psid_run_ts"]
)

SESSIONS = []
for _name in _SESSION_NAMES:
    _var, _ts = discover_session_run(results_root, "psid", EXP_BEHAVIORAL, _name)
    if not _var:
        continue
    _pid, _sess = _name.split("_S")
    SESSIONS.append(
        SessionSpec(
            label=_name,
            participant=_pid,
            session=int(_sess),
            psid_variant=_var,
            psid_run_ts=_ts,
        )
    )

In [ ]:
from collections import defaultdict

p2_root = Path("data/participants_2")


def _p2_block_info(pid, ses):
    """Read participants_2: {block: {'trials': [...], 'frag': bool, 'stim': str}}."""
    ses_path = p2_root / f"participant_id={pid}" / f"session={ses}"
    info = {}
    for bd in ses_path.glob("block=*"):
        bnum = int(bd.name.split("=")[1])
        pf = list(bd.glob("*.parquet"))
        if not pf:
            continue
        df = pl.read_parquet(pf[0])
        trials_val = df["trials"][0]
        trials_list = (
            trials_val.to_list() if hasattr(trials_val, "to_list") else list(trials_val)
        )
        frag = bool(df["is_fragmented"][0]) if "is_fragmented" in df.columns else False
        stim = str(df["stim"][0]) if "stim" in df.columns else "?"
        info[bnum] = {"trials": trials_list, "frag": frag, "stim": stim}
    return info


def _split_block_trials(variant):
    """Read split parquets: {block: {trial: stim}}."""
    framework = variant.split("_")[0]
    base = results_root / framework / variant / "split"
    bt = defaultdict(dict)
    for split in ("train", "val", "test"):
        df = pl.read_parquet(base / f"{split}.parquet")
        for r in df.select("block", "trial", "stim").iter_rows(named=True):
            bt[int(r["block"])][int(r["trial"])] = str(r["stim"])
    return bt


summary_rows = []
all_removal_details = []
dbs_dist_rows = []
frag_total, plateau_total, protocol_total = 0, 0, 0

for s in SESSIONS:
    p2 = _p2_block_info(s.participant, s.session)
    splits = _split_block_trials(s.psid_variant)
    all_expected = set(range(1, 13))
    present_blocks = set(p2.keys())
    missing_no_data = sorted(all_expected - present_blocks)

    n_frag, n_plateau, n_protocol = 0, 0, 0
    frag_blocks = []
    p2_off, p2_on, split_off, split_on = 0, 0, 0, 0

    print(f"\n{'='*60}")
    print(f"  {s.label}  ({s.participant} session {s.session})")
    print(f"{'='*60}")

    if missing_no_data:
        print(f"  Blocks with NO recording: {missing_no_data}")

    for b in sorted(present_blocks):
        info = p2[b]
        p2_trials = set(info["trials"])
        s_trial_dict = splits.get(b, {})
        s_trials = set(s_trial_dict.keys())
        dbs = "DBS-" + info["stim"].upper()

        if info["stim"].lower() == "off":
            p2_off += len(p2_trials)
        else:
            p2_on += len(p2_trials)
        for stim_val in s_trial_dict.values():
            if stim_val in ("off", "0"):
                split_off += 1
            else:
                split_on += 1

        if info["frag"]:
            frag_blocks.append(b)
            n_frag += len(p2_trials)
            print(
                f"  Block {b:2d} ({dbs}): FRAGMENTED — all {len(p2_trials)} trials dropped"
            )
            for t in sorted(p2_trials):
                all_removal_details.append(
                    {
                        "participant": s.participant,
                        "session": s.session,
                        "block": b,
                        "trial": t,
                        "dbs": dbs,
                        "reason": "fragmented",
                    }
                )
            continue

        expected_full = set(range(1, 13))
        protocol_removed = sorted(expected_full - p2_trials)
        if protocol_removed:
            n_protocol += len(protocol_removed)
            for t in protocol_removed:
                all_removal_details.append(
                    {
                        "participant": s.participant,
                        "session": s.session,
                        "block": b,
                        "trial": t,
                        "dbs": dbs,
                        "reason": "protocol/events",
                    }
                )

        plateau_removed = sorted(p2_trials - s_trials)
        if plateau_removed:
            n_plateau += len(plateau_removed)
            for t in plateau_removed:
                all_removal_details.append(
                    {
                        "participant": s.participant,
                        "session": s.session,
                        "block": b,
                        "trial": t,
                        "dbs": dbs,
                        "reason": "plateau (>2.0s)",
                    }
                )

        parts = []
        if protocol_removed:
            parts.append(f"protocol={protocol_removed}")
        if plateau_removed:
            parts.append(f"plateau={plateau_removed}")
        status = "; ".join(parts) if parts else "ok"
        print(
            f"  Block {b:2d} ({dbs}): p2={len(p2_trials)}, splits={len(s_trials)}  [{status}]"
        )

    p2_total_all = sum(len(p2[b]["trials"]) for b in p2)
    p2_total_nonfrag = sum(len(p2[b]["trials"]) for b in p2 if not p2[b]["frag"])
    split_total = sum(len(splits[b]) for b in splits)
    summary_rows.append(
        {
            "Session": s.label,
            "Blocks (p2)": len(p2),
            "Fragmented": len(frag_blocks),
            "Blocks (splits)": len(splits),
            "p2 trials": p2_total_all,
            "Split trials": split_total,
            "Protocol removed": n_protocol,
            "Plateau removed": n_plateau,
            "Fragmented removed": n_frag,
        }
    )
    dbs_dist_rows.append(
        {
            "Session": s.label,
            "p2 OFF": p2_off,
            "p2 ON": p2_on,
            "p2 total": p2_off + p2_on,
            "split OFF": split_off,
            "split ON": split_on,
            "split total": split_total,
            "removed OFF": p2_off - split_off,
            "removed ON": p2_on - split_on,
        }
    )
    frag_total += n_frag
    plateau_total += n_plateau
    protocol_total += n_protocol

# --- 1. Removal summary ---
print(f"\n{'='*60}")
print("  REMOVAL SUMMARY")
print(f"{'='*60}")
print(pl.DataFrame(summary_rows))
print(
    f"\nTotals: {protocol_total} protocol, "
    f"{plateau_total} plateau (max_pause > 2.0 s), "
    f"{frag_total} fragmented"
)

# --- 2. DBS condition distribution per session ---
print(f"\n{'='*60}")
print("  DBS CONDITION DISTRIBUTION (per session)")
print(f"{'='*60}")
print(pl.DataFrame(dbs_dist_rows))

# --- 3. Removals by DBS condition and reason ---
if all_removal_details:
    removal_df = pl.DataFrame(all_removal_details)

    print(f"\n{'='*60}")
    print("  REMOVALS BY REASON AND DBS CONDITION")
    print(f"{'='*60}")
    grand = (
        removal_df.group_by("reason", "dbs")
        .len()
        .sort("reason", "dbs")
        .rename({"len": "count"})
    )
    print(grand)

    per_session = (
        removal_df.group_by("participant", "session", "dbs", "reason")
        .len()
        .sort("participant", "session", "dbs", "reason")
        .rename({"len": "count"})
    )
    print(f"\n  Per session:")
    print(per_session)

    # --- 4. Full list of every removed trial ---
    print(f"\n{'='*60}")
    print(f"  ALL {len(removal_df)} REMOVED TRIALS")
    print(f"{'='*60}")
    with pl.Config(tbl_rows=100):
        print(removal_df.sort("participant", "session", "block", "trial"))

In [ ]:
# Split manifest CSV — derived from split parquets (no configs/splits YAML needed).
splits_cfg_dir = None  # unused, kept for reference
manifest_rows = []
for s in SESSIONS:
    fw = s.psid_variant.split("_")[0]
    base = results_root / fw / s.psid_variant / "split"
    split_dfs = {
        sp: pl.read_parquet(base / f"{sp}.parquet", columns=["block", "stim"])
        for sp in ("train", "val", "test")
    }

    def _blocks(sp):
        return sorted(split_dfs[sp]["block"].unique().to_list())

    def _n(sp, stim):
        return split_dfs[sp].filter(pl.col("stim") == stim).height

    manifest_rows.append(
        {
            "session": s.label,
            "strategy": "balanced_block_chronological",
            "train_blocks": "|".join(str(b) for b in _blocks("train")),
            "val_blocks": "|".join(str(b) for b in _blocks("val")),
            "test_blocks": "|".join(str(b) for b in _blocks("test")),
            "train_off": _n("train", "off"),
            "train_on": _n("train", "on"),
            "val_off": _n("val", "off"),
            "val_on": _n("val", "on"),
            "test_off": _n("test", "off"),
            "test_on": _n("test", "on"),
        }
    )
if manifest_rows:
    manifest_df = pl.DataFrame(manifest_rows)
    manifest_csv = OUT / "split_manifest.csv"
    manifest_df.write_csv(str(manifest_csv))
    print(f"Wrote split manifest -> {manifest_csv}")
    print(manifest_df)

In [ ]:
# Enrich removal details with session label
_sess_lookup = {(s.participant, s.session): s.label for s in SESSIONS}
for d in all_removal_details:
    d["label"] = _sess_lookup.get((d["participant"], d["session"]), "")

# --- Collect per-split DBS counts --- derived from split parquets directly.
split_dbs_rows = []
for s in SESSIONS:
    fw = s.psid_variant.split("_")[0]
    base = results_root / fw / s.psid_variant / "split"
    for split_name in ("train", "val", "test"):
        df = pl.read_parquet(base / f"{split_name}.parquet", columns=["stim"])
        n_off = df.filter(pl.col("stim") == "off").height
        n_on = df.filter(pl.col("stim") == "on").height
        split_dbs_rows.append(
            {
                "session": s.label,
                "split": split_name,
                "OFF": n_off,
                "ON": n_on,
                "total": n_off + n_on,
            }
        )
split_dbs_df = pl.DataFrame(split_dbs_rows)
sessions = [s.label for s in SESSIONS]

session_totals = {
    sl: int(split_dbs_df.filter(pl.col("session") == sl)["total"].sum())
    for sl in sessions
}

split_order = ["train", "val", "test"]
SPLIT_COLOR = {"train": "#D32F2F", "val": "#1976D2", "test": "#616161"}
DBS_ALPHA = {"OFF": 0.55, "ON": 1.00}

reason_meta = [
    ("fragmented", "#E24B4A"),
    ("protocol/events", "#888780"),
    ("plateau (>2.0s)", COLOR_DBS_OFF),
]


def _split_dbs_values(sp, dbs, as_fraction):
    out = []
    for sl in sessions:
        r = split_dbs_df.filter((pl.col("session") == sl) & (pl.col("split") == sp))
        c = r[dbs][0] if r.height else 0
        if as_fraction:
            out.append(c / session_totals[sl] if session_totals[sl] else 0)
        else:
            out.append(c)
    return np.array(out, dtype=float)


# ── Figure 1: Trials removed by reason ───────────────────────────────────
fig_a, ax_a = plt.subplots(figsize=(6.5, 3.2))
bottoms = np.zeros(len(sessions))
for reason, color in reason_meta:
    counts = np.array(
        [
            sum(
                1
                for d in all_removal_details
                if d["label"] == sl and d["reason"] == reason
            )
            for sl in sessions
        ],
        dtype=float,
    )
    bars = ax_a.bar(
        sessions, counts, bottom=bottoms, color=color, width=0.55, label=reason
    )
    stack_bar_label(ax_a, bars, fmt="{:.0f}", min_value=1)
    bottoms += counts
ax_a.set_ylabel("Trials")
ax_a.set_ylim(bottom=0)
ax_a.legend(title="Removal reason")
panel_label(ax_a, "A", "Trials removed by reason")
fig_a.savefig(str(OUT / "fig_trials_removed.png"))
plt.show()

# ── Figure 2: Split ratios with DBS condition ─────────────────────────────
DBS_SERIES = [("OFF", None), ("ON", None)]

fig_b, ax_b = plt.subplots(figsize=(6.5, 3.5))
fig_b.set_constrained_layout(False)
bottoms = np.zeros(len(sessions))
for sp in split_order:
    for dbs, _ in DBS_SERIES:
        vals_frac = _split_dbs_values(sp, dbs, as_fraction=True)
        vals_abs = _split_dbs_values(sp, dbs, as_fraction=False)
        bars = ax_b.bar(
            sessions,
            vals_frac,
            bottom=bottoms,
            color=hex_to_rgba(SPLIT_COLOR[sp], DBS_ALPHA[dbs]),
            width=0.55,
            label=f"{sp} DBS-{dbs}",
        )
        stack_bar_label(ax_b, bars, values=vals_abs, fmt="{:.0f}", min_value=0.02)
        bottoms += vals_frac
ax_b.set_ylabel("Split ratio")
ax_b.set_ylim(0, 1.05)
panel_label(ax_b, "B", "Split ratio with DBS condition")
_h, _l = ax_b.get_legend_handles_labels()
_order = [0, 2, 4, 1, 3, 5]
fig_b.legend(
    [_h[i] for i in _order], [_l[i] for i in _order], ncol=3, loc="lower center"
)
fig_b.subplots_adjust(bottom=0.22)
fig_b.savefig(str(OUT / "fig_split_ratio.png"))
plt.show()

# ── Figure 3: Trial counts per split ─────────────────────────────────────
fig_c, ax_c = plt.subplots(figsize=(6.5, 3.5))
fig_c.set_constrained_layout(False)
bottoms = np.zeros(len(sessions))
for sp in split_order:
    for dbs, _ in DBS_SERIES:
        vals = _split_dbs_values(sp, dbs, as_fraction=False)
        bars = ax_c.bar(
            sessions,
            vals,
            bottom=bottoms,
            color=hex_to_rgba(SPLIT_COLOR[sp], DBS_ALPHA[dbs]),
            width=0.55,
            label=f"{sp} DBS-{dbs}",
        )
        stack_bar_label(ax_c, bars, fmt="{:.0f}", min_value=1)
        bottoms += vals
ax_c.set_ylabel("Trials")
panel_label(ax_c, "C", "Trial counts per split")
_h, _l = ax_c.get_legend_handles_labels()
_order = [0, 2, 4, 1, 3, 5]
fig_c.legend(
    [_h[i] for i in _order], [_l[i] for i in _order], ncol=3, loc="lower center"
)
fig_c.subplots_adjust(bottom=0.22)
fig_c.savefig(str(OUT / "fig_split_counts.png"))
plt.show()

## PSD DBS comparison — all channels per session

In [ ]:
from scipy.signal import welch
from scipy.stats import mannwhitneyu
import matplotlib.cm as cm
from matplotlib.lines import Line2D

FS_SPLIT = 200
MARGIN_S = 2
MARGIN_SAMP = MARGIN_S * FS_SPLIT
FREQ_CUTOFF = 85

ECOG_CHANNELS = [1, 2, 3, 4]
LAP_CHANNEL_NAMES = ["8-10", "9-11", "10-12", "11-13", "12-14", "13-15", "14-16"]

ECOG_COLORS = [cm.Blues(v) for v in np.linspace(0.40, 0.85, len(ECOG_CHANNELS))]
LAP_COLORS = [cm.Oranges(v) for v in np.linspace(0.35, 0.90, len(LAP_CHANNEL_NAMES))]
ALPHA_OFF = 0.35


def _session_raw_files(participant, session):
    return sorted(
        (raw_data_root / f"participant_id={participant}" / f"session={session}").glob(
            "*/*.parquet"
        )
    )


_sample_paths = _session_raw_files(SESSIONS[0].participant, SESSIONS[0].session)
_sample_df = pl.read_parquet(_sample_paths[0], n_rows=1)

_psd_cols_ecog = {
    ch: sorted(
        [
            c
            for c in _sample_df.columns
            if c.startswith(f"ECOG_{ch}_") and c.endswith("_raw")
        ]
    )
    for ch in ECOG_CHANNELS
}
_psd_cols_lap = {
    name: sorted(
        [
            c
            for c in _sample_df.columns
            if c.startswith(f"LAPLACIAN_{name}_") and c.endswith("_raw")
        ]
    )
    for name in LAP_CHANNEL_NAMES
}

# Frequency axis from first real trial
_freqs_ref = None
_tmp_cols = _psd_cols_ecog[1]
for _fp in _sample_paths:
    _df = pl.read_parquet(_fp, columns=["stim"] + _tmp_cols)
    for _row in _df.iter_rows(named=True):
        _sig = np.zeros(len(_row[_tmp_cols[0]]))
        for _c in _tmp_cols:
            _sig += np.array(_row[_c], dtype=float)
        _sig = _sig[MARGIN_SAMP:-MARGIN_SAMP]
        _sig = _sig[np.isfinite(_sig)]
        if len(_sig) < 400:
            continue
        _nperseg = min(len(_sig), FS_SPLIT * 2)
        _freqs_ref, _ = welch(
            _sig, fs=FS_SPLIT, nperseg=_nperseg, noverlap=_nperseg // 2
        )
        break
    if _freqs_ref is not None:
        break

freq_mask = _freqs_ref <= FREQ_CUTOFF
freqs_plot = _freqs_ref[freq_mask]


def _compute_session_psds(cols_dict, keys, sessions):
    """Mean trial PSD per condition for each channel key x session.

    Returns {key: {session_label: {'on': arr|None, 'off': arr|None}}}.
    """
    raw = {key: {s.label: {"off": [], "on": []} for s in sessions} for key in keys}
    for key in keys:
        cols = cols_dict[key]
        for s in sessions:
            for fp in _session_raw_files(s.participant, s.session):
                df = pl.read_parquet(fp, columns=["stim"] + cols)
                for row in df.iter_rows(named=True):
                    cond = "off" if row["stim"] in ("off", "0") else "on"
                    sig = np.zeros(len(row[cols[0]]))
                    for c in cols:
                        sig += np.array(row[c], dtype=float)
                    sig = sig[MARGIN_SAMP:-MARGIN_SAMP]
                    sig = sig[np.isfinite(sig)]
                    if len(sig) < 400:
                        continue
                    nperseg = min(len(sig), FS_SPLIT * 2)
                    _, psd = welch(
                        sig, fs=FS_SPLIT, nperseg=nperseg, noverlap=nperseg // 2
                    )
                    raw[key][s.label][cond].append(10 * np.log10(psd + 1e-20))
    result = {}
    for key in keys:
        result[key] = {}
        for sl in raw[key]:
            result[key][sl] = {}
            for cond in ("off", "on"):
                trials = raw[key][sl][cond]
                if trials:
                    mat = np.vstack([p[freq_mask] for p in trials])
                    result[key][sl][cond] = mat.mean(axis=0)
                else:
                    result[key][sl][cond] = None
    return result


print("Computing ECoG PSDs...")
psds_ecog = _compute_session_psds(_psd_cols_ecog, ECOG_CHANNELS, SESSIONS)
print("Computing Laplacian PSDs...")
psds_lap = _compute_session_psds(_psd_cols_lap, LAP_CHANNEL_NAMES, SESSIONS)
print("Done.")

## PSD DBS comparison — per session, ECoG + Laplacian overlaid

Each session: left panel = all 4 ECoG channels; right = all 7 Laplacian channels.
Solid lines = DBS-ON (mean across trials), faded = DBS-OFF. Blue shades = ECoG; orange shades = Laplacian.

In [ ]:
# Per-session PSD: all ECoG channels (left) + all Laplacian channels (right).
_SESSION_FIG_IDX = {"PDI1_S2": 2, "PDI1_S4": 3, "PDI4_S2": 4, "PDI4_S3": 5}

for s in SESSIONS:
    fig, (ax_ecog, ax_lap) = plt.subplots(1, 2, figsize=(9.0, 4.0))
    fig.set_constrained_layout(False)

    # ECoG — 4 channels, blue shades; ON full alpha, OFF faded; all solid
    for ch_idx, ch in enumerate(ECOG_CHANNELS):
        color = ECOG_COLORS[ch_idx]
        m_on = psds_ecog[ch][s.label]["on"]
        m_off = psds_ecog[ch][s.label]["off"]
        if m_on is not None:
            ax_ecog.plot(
                freqs_plot, m_on, color=color, lw=0.9, ls="-", label=f"ECOG_{ch}"
            )
        if m_off is not None:
            ax_ecog.plot(
                freqs_plot, m_off, color=color, lw=0.9, ls="-", alpha=ALPHA_OFF
            )

    ax_ecog.set_xlim(0, FREQ_CUTOFF)
    ax_ecog.set_xticks(range(0, FREQ_CUTOFF + 1, 10))
    ax_ecog.set_xlabel("Frequency (Hz)")
    ax_ecog.set_ylabel("PSD (dB/Hz)")
    panel_label(ax_ecog, "A", f"ECoG - {s.label}")

    # Laplacian — 7 channels, orange shades; ON full, OFF faded; all solid
    for lap_idx, lap_name in enumerate(LAP_CHANNEL_NAMES):
        color = LAP_COLORS[lap_idx]
        m_on = psds_lap[lap_name][s.label]["on"]
        m_off = psds_lap[lap_name][s.label]["off"]
        if m_on is not None:
            ax_lap.plot(freqs_plot, m_on, color=color, lw=0.9, ls="-", label=lap_name)
        if m_off is not None:
            ax_lap.plot(freqs_plot, m_off, color=color, lw=0.9, ls="-", alpha=ALPHA_OFF)

    ax_lap.set_xlim(0, FREQ_CUTOFF)
    ax_lap.set_xticks(range(0, FREQ_CUTOFF + 1, 10))
    ax_lap.set_xlabel("Frequency (Hz)")
    ax_lap.set_ylabel("PSD (dB/Hz)")
    panel_label(ax_lap, "B", f"Laplacian LFP - {s.label}")

    # ECoG legend below panel A: channels + DBS-ON/OFF indicators
    _ch_h, _ch_l = ax_ecog.get_legend_handles_labels()
    _cond_h = [
        Line2D([0], [0], color="gray", lw=1.2, ls="-", label="DBS-ON"),
        Line2D(
            [0], [0], color="gray", lw=1.2, ls="-", alpha=ALPHA_OFF, label="DBS-OFF"
        ),
    ]
    ax_ecog.legend(
        handles=_ch_h + _cond_h,
        labels=_ch_l + ["DBS-ON", "DBS-OFF"],
        loc="upper center",
        bbox_to_anchor=(0.5, -0.18),
        ncol=3,
    )
    # Laplacian legend below panel B
    ax_lap.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=4)

    fig.subplots_adjust(left=0.08, right=0.98, top=0.92, bottom=0.32, wspace=0.30)
    fig_num = _SESSION_FIG_IDX[s.label]
    out_name = f"fig_{fig_num:03d}_psd_{s.label}.png"
    fig.savefig(str(OUT / out_name))
    plt.show()
    print(f"{s.label} -> {out_name}")

## Behavioral signals — raw trial traces (velocity, acceleration)


In [ ]:
# Behavioral raw data — z-scored per trial, mean ± SEM bands (smoothed)

FS_BEH = 200
TRIAL_SEC = 9
N_SAMP = TRIAL_SEC * FS_BEH
SMOOTH_WIN = 41
beh_vars = ["tracing_velocity_x", "tracing_acceleration_magnitude"]


def _smooth(arr, win):
    kernel = np.ones(win) / win
    padded = np.pad(arr, win // 2, mode="edge")
    return np.convolve(padded, kernel, mode="valid")[: len(arr)]


for var_name in beh_vars:
    traces_by_session = {}
    for s in SESSIONS:
        framework = s.psid_variant.split("_")[0]
        base = results_root / framework / s.psid_variant / "split"
        traces = {"off": [], "on": []}
        for split in ("train", "val", "test"):
            fp = base / f"{split}.parquet"
            if not fp.exists():
                continue
            df = pl.read_parquet(fp, columns=["stim", var_name])
            for row in df.iter_rows(named=True):
                cond = "off" if row["stim"] in ("off", "0") else "on"
                sig = np.array(row[var_name], dtype=float)[:N_SAMP]
                if len(sig) < N_SAMP:
                    continue
                mu, sd = np.nanmean(sig), np.nanstd(sig)
                if sd < 1e-8:
                    continue
                sig = (sig - mu) / sd
                sig = np.nan_to_num(sig, nan=0.0)
                traces[cond].append(sig)
        traces_by_session[s.label] = traces

    fig, axes = plt.subplots(2, 2, sharex=True, sharey=True, figsize=(7.5, 4.8))
    fig.set_constrained_layout(False)
    t_axis = np.arange(N_SAMP) / FS_BEH
    for s_idx, s in enumerate(SESSIONS):
        ax = axes.flat[s_idx]
        traces = traces_by_session[s.label]
        for cond_key, col_c, cond_label in [
            ("off", COLOR_DBS_OFF, "DBS-OFF"),
            ("on", COLOR_DBS_ON, "DBS-ON"),
        ]:
            cond_traces = traces[cond_key]
            if not cond_traces:
                continue
            mat = np.vstack(cond_traces)
            mean = _smooth(mat.mean(axis=0), SMOOTH_WIN)
            sem = _smooth(mat.std(axis=0) / np.sqrt(len(mat)), SMOOTH_WIN)
            ax.fill_between(
                t_axis, mean - sem, mean + sem, color=col_c, alpha=0.20, linewidth=0
            )
            ax.plot(t_axis, mean, color=col_c, label=cond_label)
        ax.set_xlim(0, TRIAL_SEC)
        ax.set_xticks(range(0, TRIAL_SEC + 1))
        panel_label(ax, chr(65 + s_idx), s.label)

    for ax in axes[-1, :]:
        ax.set_xlabel("Time (s)")
    for ax in axes[:, 0]:
        ax.set_ylabel("z-score")

    handles, labels = axes.flat[0].get_legend_handles_labels()
    fig.legend(handles, labels, ncol=2, loc="lower center", frameon=False)
    fig.subplots_adjust(bottom=0.12)

    safe_name = var_name.replace(" ", "_")
    fig.savefig(str(OUT / f"fig_beh_{safe_name}.png"))
    plt.show()
    print(f"{var_name} figure saved")

## Grid alignment — behavioral resampling to 200 Hz neural grid

**A** Interpolation on the common time grid (100 ms window).  
**B** x position density: raw vs aligned (KDE, pooled across sessions).  
**C** Sample tracing trajectory (single trial).

In [ ]:
# Grid alignment — three standalone figures

from scipy.stats import gaussian_kde
from modules.lib.fig_grid_alignment import _load_trial_row, _raw_and_interp_signal

GRID_DATA_ROOT = Path(
    "resampled_recordings/participants_at_200Hz_scaled_1e6_narrow_band"
)
MARGIN_SAMP = 400
# Raw behavioural = DPAD-warm, aligned = PSID-blue. Neural dot uses a brighter red
# to distinguish it from the DBS-ON red elsewhere.
RAW_COLOR, ALIGNED_COLOR, NEURAL_COLOR = COLOR_DPAD, COLOR_PSID, "#C43A31"

EX_P, EX_S, EX_T = "PDI1", "2", 30
row = _load_trial_row(
    participant=EX_P, session=EX_S, data_root=GRID_DATA_ROOT, trial_index=EX_T
)
print(
    f"Example: {EX_P} session {EX_S}, block {row.get('block')}, trial {row.get('trial')} (idx {EX_T})"
)

t_raw_beh, vel_raw, t_grid_full, vel_grid = _raw_and_interp_signal(
    row, "tracing_velocity_x"
)
valid_grid = ~np.isnan(t_grid_full)
t_grid = t_grid_full[valid_grid]

x_raw = np.asarray(row["x"], dtype=float)
y_raw = np.asarray(row["y"], dtype=float)
t_motion = np.asarray(row["motion_time"], dtype=float)
order = np.argsort(t_motion)
x_raw, y_raw, t_motion = x_raw[order], y_raw[order], t_motion[order]

print(f"  ECOG_1_alpha: {len(t_grid)} pts (200 Hz)")
print(
    f"  tracing_velocity_x (raw): {len(t_raw_beh)} pts (~{1.0/np.median(np.diff(np.sort(t_raw_beh))):.0f} Hz)"
)
print(f"  tracing_velocity_x (aligned): {len(t_grid)} pts (200 Hz)")
print(
    f"  Tracing path: x [{np.nanmin(x_raw):.0f}, {np.nanmax(x_raw):.0f}], y [{np.nanmin(y_raw):.0f}, {np.nanmax(y_raw):.0f}]"
)

mid = 0.5 * (t_grid.min() + t_grid.max())
WIN_S = 0.100
win_lo, win_hi = mid - WIN_S / 2, mid + WIN_S / 2
mg = (t_grid >= win_lo) & (t_grid <= win_hi)
tg_w = (t_grid[mg] - win_lo) * 1000
mr = (t_raw_beh >= win_lo) & (t_raw_beh <= win_hi)
tr_w = (t_raw_beh[mr] - win_lo) * 1000
print(f"  Window: {WIN_S*1000:.0f} ms \u2014 {len(tg_w)} grid, {len(tr_w)} raw")

# Pooled distributions
all_x_raw, all_x_interp = [], []
for s in SESSIONS:
    gp = str(
        GRID_DATA_ROOT
        / f"participant_id={s.participant}"
        / f"session={s.session}"
        / "*"
        / "*.parquet"
    )
    df = pl.read_parquet(gp, columns=["motion_time", "x", "time_original"])
    for idx in range(df.height):
        r = df.row(idx, named=True)
        t_r = np.asarray(r["motion_time"], dtype=float)
        x_r = np.asarray(r["x"], dtype=float)
        t_g = np.asarray(r["time_original"], dtype=float)
        o = np.argsort(t_r)
        t_r, x_r = t_r[o], x_r[o]
        m = ~np.isnan(x_r) & ~np.isnan(t_r)
        if m.sum() < 10:
            continue
        t_r, x_r = t_r[m], x_r[m]
        vg = ~np.isnan(t_g)
        t_g = t_g[vg]
        all_x_raw.append(x_r)
        all_x_interp.append(np.interp(t_g, t_r, x_r))
all_x_raw = np.concatenate(all_x_raw)
all_x_interp = np.concatenate(all_x_interp)
print(
    f"Pooled: {len(all_x_raw)} raw, {len(all_x_interp)} interp ({', '.join(s.label for s in SESSIONS)})"
)

# ── A: Interpolation on the common time grid ──
fig_a, ax_a = plt.subplots(figsize=(9, 3.2))
for t_tick in tg_w:
    ax_a.axvline(
        t_tick, color=(0.627, 0.627, 0.608), alpha=0.30, linewidth=0.9, zorder=1
    )
ax_a.scatter(
    tg_w,
    np.full(len(tg_w), 2.0),
    color=NEURAL_COLOR,
    s=40,
    label="ECOG_1_alpha (200 Hz)",
    zorder=3,
)
ax_a.scatter(
    tr_w,
    np.full(len(tr_w), 1.0),
    color=RAW_COLOR,
    s=40,
    label="tracing_velocity_x raw (~119 Hz)",
    zorder=3,
)
ax_a.scatter(
    tg_w,
    np.full(len(tg_w), 0.0),
    color=ALIGNED_COLOR,
    s=40,
    label="tracing_velocity_x aligned (200 Hz)",
    zorder=3,
)
ax_a.set_yticks([0, 1, 2])
ax_a.set_yticklabels(
    ["tracing_velocity_x\n(aligned)", "tracing_velocity_x\n(raw)", "ECOG_1_alpha"]
)
ax_a.set_ylim(-0.5, 2.5)
ax_a.set_xlim(tg_w.min() - 1, tg_w.max() + 1)
ax_a.set_xlabel("Time (ms)")
panel_label(ax_a, "A", "Interpolation on the common time grid")
fig_a.savefig(str(OUT / "fig_grid_alignment_A.png"))
plt.show()
print("Saved: fig_grid_alignment_A.png")

# ── B: x position density: raw vs aligned ──
fig_b, ax_b = plt.subplots(figsize=(9, 3.3))
lo_clip = np.percentile(np.concatenate([all_x_raw, all_x_interp]), 0.5)
hi_clip = np.percentile(np.concatenate([all_x_raw, all_x_interp]), 99.5)
kde_x_pts = np.linspace(lo_clip, hi_clip, 400)
raw_clip = all_x_raw[(all_x_raw >= lo_clip) & (all_x_raw <= hi_clip)]
interp_clip = all_x_interp[(all_x_interp >= lo_clip) & (all_x_interp <= hi_clip)]
kde_raw = gaussian_kde(raw_clip, bw_method=0.06)
kde_int = gaussian_kde(interp_clip, bw_method=0.06)
raw_y = kde_raw(kde_x_pts)
int_y = kde_int(kde_x_pts)
ax_b.fill_between(kde_x_pts, 0, raw_y, color=RAW_COLOR, alpha=0.12)
ax_b.plot(kde_x_pts, raw_y, color=RAW_COLOR, linewidth=2.0, label="x position (raw)")
ax_b.plot(
    kde_x_pts,
    int_y,
    color=ALIGNED_COLOR,
    linewidth=2.0,
    linestyle="--",
    label="x position (interpolated)",
)
ax_b.set_xlabel("x position (px)")
ax_b.set_ylabel("Density")
ax_b.legend(frameon=False)
panel_label(ax_b, "B", "x position density: raw vs aligned")
fig_b.savefig(str(OUT / "fig_grid_alignment_B.png"))
plt.show()
print("Saved: fig_grid_alignment_B.png")

# ── C: Sample tracing trajectory ──
fig_c, ax_c = plt.subplots(figsize=(3.3, 3.3))
ax_c.plot(x_raw, y_raw, color=RAW_COLOR, linewidth=1.5)
ax_c.set_xlabel("x (px)")
ax_c.set_ylabel("y (px)")
ax_c.set_aspect("equal", adjustable="datalim")
panel_label(ax_c, "C", "Sample tracing trajectory")
fig_c.savefig(str(OUT / "fig_grid_alignment_C.png"))
plt.show()
print("Saved: fig_grid_alignment_C.png")